In [7]:
import pandas as pd
import numpy as np
import re

# Đọc và chuẩn hóa số lượng cột bằng Python thuần
cleaned_lines = []
expected_fields = 10

with open("patient_heart_rate.csv", "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(",")
        if len(parts) == 11:
            # Gộp Firstname và Lastname bị phân tách bởi dấu phẩy lỗi
            merged_name = parts[1].strip() + " " + parts[2].strip()
            new_line = [parts[0], merged_name] + parts[3:]
            cleaned_lines.append(new_line)
        elif len(parts) == 10:
            cleaned_lines.append(parts)
        else:
            while len(parts) < expected_fields: parts.append("")
            cleaned_lines.append(parts[:expected_fields])

# Tạo DataFrame
column_names = ["Id", "Name", "Age", "Weight", "m0006", "m0612", "m1218", "f0006", "f0612", "f1218"]
df = pd.DataFrame(cleaned_lines, columns=column_names)

# Ép kiểu dữ liệu số cơ bản
df['Id'] = pd.to_numeric(df['Id'], errors='coerce')
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# Tách cột Name thành Firstname và Lastname
df[['Firstname', 'Lastname']] = df['Name'].str.split(expand=True, n=1)
df.drop(columns=['Name'], inplace=True, errors='ignore')
df['Firstname'].fillna('', inplace=True)
df['Lastname'].fillna('', inplace=True)

print("--- Đã hoàn thành Đoạn 1: Tải dữ liệu an toàn ---")
print(df.head())

--- Đã hoàn thành Đoạn 1: Tải dữ liệu an toàn ---
    Id   Age      Weight m0006 m0612 m1218 f0006 f0612 f1218 Firstname  \
0  NaN  56.0       70kgs    72    69    71     -     -     -    Mickéy   
1  2.0  34.0   154.89lbs     -     -     -    85    84    76    Donald   
2  3.0  16.0                 -     -     -    65    69    72      Mini   
3  4.0   NaN       78kgs    78    79    72     -     -     -   Scrooge   
4  5.0  54.0  198.658lbs     -     -     -    69          75      Pink   

  Lastname  
0    Mousé  
1     Duck  
2    Mouse  
3   McDuck  
4  Panther  


/tmp/ipykernel_2965/760833102.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Firstname'].fillna('', inplace=True)
/tmp/ipykernel_2965/760833102.py:38: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'd

In [8]:
def clean_weight(value):
    if pd.isna(value) or str(value).strip() == '':
        return np.nan
    val_str = str(value).strip().lower()
    if "lbs" in val_str:
        num = float(val_str.replace("lbs", "").strip())
        return f"{int(num / 2.2)}kgs"
    elif "kgs" in val_str:
        return val_str
    else:
        cleaned_num = re.sub(r'[^0-9.]', '', val_str)
        if cleaned_num:
            return f"{int(float(cleaned_num))}kgs"
    return np.nan

df['Weight'] = df['Weight'].apply(clean_weight)
print("--- Đã hoàn thành Đoạn 2: Chuẩn hóa cột Weight ---")
print(df[['Firstname', 'Weight']].head())

--- Đã hoàn thành Đoạn 2: Chuẩn hóa cột Weight ---
  Firstname Weight
0    Mickéy  70kgs
1    Donald  70kgs
2      Mini    NaN
3   Scrooge  78kgs
4      Pink  90kgs


In [9]:
# Vấn đề 4: Xóa dòng trống hoàn toàn
df.dropna(how="all", inplace=True)

# Vấn đề 5: Xóa trùng lặp thông tin cá nhân
df.drop_duplicates(subset=['Firstname', 'Lastname', 'Age', 'Weight'], inplace=True)

# Vấn đề 6: Xóa ký tự non-ASCII
df['Firstname'] = df['Firstname'].str.replace(r'[^\x00-\x7F]+', '', regex=True)
df['Lastname'] = df['Lastname'].str.replace(r'[^\x00-\x7F]+', '', regex=True)

print("--- Đã hoàn thành Đoạn 3: Làm sạch hàng trùng lặp và ký tự lạ ---")

--- Đã hoàn thành Đoạn 3: Làm sạch hàng trùng lặp và ký tự lạ ---


In [10]:
print(f"Số dòng thiếu Age ban đầu: {df['Age'].isna().sum()}")
print(f"Số dòng thiếu Weight ban đầu: {df['Weight'].isna().sum()}")

# Điều kiện lọc: Trống cả hai thì xóa
df.dropna(subset=['Age', 'Weight'], how='all', inplace=True)

# Điền giá trị trung bình cột tuổi vào các ô trống
mean_age = df['Age'].mean()
df['Age'].fillna(mean_age, inplace=True)

print("--- Đã hoàn thành Đoạn 4: Xử lý khuyết thiếu Age và Weight ---")

Số dòng thiếu Age ban đầu: 5
Số dòng thiếu Weight ban đầu: 5
--- Đã hoàn thành Đoạn 4: Xử lý khuyết thiếu Age và Weight ---


/tmp/ipykernel_2965/2947724338.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(mean_age, inplace=True)


In [11]:
heart_cols = ["m0006", "m0612", "m1218", "f0006", "f0612", "f1218"]
df_tidy = pd.melt(
    df,
    id_vars=['Id', 'Age', 'Weight', 'Firstname', 'Lastname'],
    value_vars=heart_cols,
    var_name='sex_and_time',
    value_name='PulseRate'
)

# Tách chuỗi quy đổi sang Sex và Time bằng Regex
parsed = df_tidy['sex_and_time'].str.extract(r'([mf])(\d{2})(\d{2})')
df_tidy['Sex'] = parsed[0]
df_tidy['Time'] = parsed[1] + '-' + parsed[2]
df_tidy.drop(columns=['sex_and_time'], inplace=True)

# Chuẩn hóa dữ liệu cột PulseRate (Nhịp tim) về dạng số
df_tidy['PulseRate'] = df_tidy['PulseRate'].astype(str).str.strip()
df_tidy['PulseRate'] = df_tidy['PulseRate'].replace(['nan', '-', '', 'NaN', 'None'], np.nan)
df_tidy['PulseRate'] = pd.to_numeric(df_tidy['PulseRate'], errors='coerce')

# Sắp xếp lại cấu trúc dữ liệu theo Id tăng dần
df_tidy.sort_values(by=['Id', 'Sex', 'Time'], inplace=True)
df_tidy.reset_index(drop=True, inplace=True)

print("--- Đã hoàn thành Đoạn 5: Biến đổi cấu trúc bảng dọc (Melt) ---")
print(df_tidy.head(10))

--- Đã hoàn thành Đoạn 5: Biến đổi cấu trúc bảng dọc (Melt) ---
    Id   Age Weight Firstname Lastname  PulseRate Sex   Time
0  2.0  34.0  70kgs    Donald     Duck       85.0   f  00-06
1  2.0  34.0  70kgs    Donald     Duck       84.0   f  06-12
2  2.0  34.0  70kgs    Donald     Duck       76.0   f  12-18
3  2.0  34.0  70kgs    Donald     Duck        NaN   m  00-06
4  2.0  34.0  70kgs    Donald     Duck        NaN   m  06-12
5  2.0  34.0  70kgs    Donald     Duck        NaN   m  12-18
6  3.0  16.0    NaN      Mini    Mouse       65.0   f  00-06
7  3.0  16.0    NaN      Mini    Mouse       69.0   f  06-12
8  3.0  16.0    NaN      Mini    Mouse       72.0   f  12-18
9  3.0  16.0    NaN      Mini    Mouse        NaN   m  00-06


In [12]:
missing_pulse_ratio = df_tidy['PulseRate'].isna().mean() * 100
print(f"Tỉ lệ thiếu hụt dữ liệu huyết áp: {missing_pulse_ratio:.2f}%")

# Chuẩn bị dữ liệu trung bình cho các phương án dự phòng
global_mean = df_tidy['PulseRate'].mean()
sex_mean = df_tidy.groupby('Sex')['PulseRate'].transform('mean')
user_mean = df_tidy.groupby('Id')['PulseRate'].transform('mean')

# Tạo các cột giá trị tịnh tiến (Shift) để lấy giá trị lân cận
df_tidy['prev1'] = df_tidy.groupby('Id')['PulseRate'].shift(1)
df_tidy['prev2'] = df_tidy.groupby('Id')['PulseRate'].shift(2)
df_tidy['next1'] = df_tidy.groupby('Id')['PulseRate'].shift(-1)
df_tidy['next2'] = df_tidy.groupby('Id')['PulseRate'].shift(-2)

final_pulse = df_tidy['PulseRate'].copy()

# Chạy thuật toán nội suy rà soát từng dòng
for i in range(len(df_tidy)):
    if pd.isna(final_pulse[i]):
        # 1. Trung bình của liền trước và liền sau
        if pd.notna(df_tidy['prev1'][i]) and pd.notna(df_tidy['next1'][i]):
            final_pulse[i] = (df_tidy['prev1'][i] + df_tidy['next1'][i]) / 2
        # 2. Trung bình của 2 giá trị liền trước
        elif pd.notna(df_tidy['prev1'][i]) and pd.notna(df_tidy['prev2'][i]):
            final_pulse[i] = (df_tidy['prev1'][i] + df_tidy['prev2'][i]) / 2
        # 3. Trung bình của 2 giá trị liền sau
        elif pd.notna(df_tidy['next1'][i]) and pd.notna(df_tidy['next2'][i]):
            final_pulse[i] = (df_tidy['next1'][i] + df_tidy['next2'][i]) / 2
        # 4. Trung bình huyết áp của chính người đó
        elif pd.notna(user_mean[i]):
            final_pulse[i] = user_mean[i]
        # 5. Trung bình huyết áp của nhóm giới tính
        elif pd.notna(sex_mean[i]):
            final_pulse[i] = sex_mean[i]
        # 6. Giá trị mặc định y khoa ổn định
        else:
            final_pulse[i] = global_mean if pd.notna(global_mean) else 72

df_tidy['PulseRate'] = final_pulse

# Dọn dẹp các cột phụ trợ, làm tròn số liệu
df_tidy.drop(columns=['prev1', 'prev2', 'next1', 'next2'], inplace=True)
df_tidy['Age'] = df_tidy['Age'].round(1)
df_tidy['PulseRate'] = df_tidy['PulseRate'].round(1)

# Loại bỏ dòng trống PulseRate tuyệt đối (nếu có) và Reindex
df_tidy.dropna(subset=['PulseRate'], inplace=True)
df_tidy.reset_index(drop=True, inplace=True)

# LƯU FILE KẾT QUẢ
output_file = 'patient_heart_rate_clean.csv'
df_tidy.to_csv(output_file, index=False)

print(f"\n--- Đã hoàn thành Đoạn 6: Xuất file sạch '{output_file}' thành công! ---")
print(df_tidy.head(15))

Tỉ lệ thiếu hụt dữ liệu huyết áp: 51.39%

--- Đã hoàn thành Đoạn 6: Xuất file sạch 'patient_heart_rate_clean.csv' thành công! ---
     Id   Age Weight Firstname Lastname  PulseRate Sex   Time
0   2.0  34.0  70kgs    Donald     Duck       85.0   f  00-06
1   2.0  34.0  70kgs    Donald     Duck       84.0   f  06-12
2   2.0  34.0  70kgs    Donald     Duck       76.0   f  12-18
3   2.0  34.0  70kgs    Donald     Duck       80.0   m  00-06
4   2.0  34.0  70kgs    Donald     Duck       81.7   m  06-12
5   2.0  34.0  70kgs    Donald     Duck       81.7   m  12-18
6   3.0  16.0    NaN      Mini    Mouse       65.0   f  00-06
7   3.0  16.0    NaN      Mini    Mouse       69.0   f  06-12
8   3.0  16.0    NaN      Mini    Mouse       72.0   f  12-18
9   3.0  16.0    NaN      Mini    Mouse       70.5   m  00-06
10  3.0  16.0    NaN      Mini    Mouse       68.7   m  06-12
11  3.0  16.0    NaN      Mini    Mouse       68.7   m  12-18
12  4.0  36.1  78kgs   Scrooge   McDuck       76.3   f  00-06
13